# P61 — Sobre los peligros de los loros estocásticos: ¿pueden ser demasiado grandes los modelos de lenguaje?

## 1. Título y paper

**Paper:** *On the Dangers of Stochastic Parrots: Can Language Models Be Too Big?*  
**Autoría:** Emily M. Bender, Timnit Gebru, Angelina McMillan-Major, Shmargaret Shmitchell  
**Año y venue:** 2021 · FAccT '21 · ACM Conference on Fairness, Accountability, and Transparency  
**Nivel:** L1 · **Motor:** `stochastic_parrots`  
**Ficha completa:** [`P61_stochastic_parrots`](../../papers/foundational/P61_stochastic_parrots/README.md)

**Hito:** Pone por escrito el coste de la carrera por el tamaño: quién paga, quién queda representado y qué se afirma de más sobre la comprensión.

- [DOI (ACM FAccT 2021)](https://doi.org/10.1145/3442188.3445922)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El crecimiento de los modelos de lenguaje se justificaba solo por la mejora en benchmarks. Los costes ambientales, la composición del corpus y las afirmaciones sobre comprensión no se auditaban.
2. Ejecutar una implementación mínima de la propuesta: Un análisis de riesgos previo al entrenamiento: documentar el corpus, contabilizar el coste, y no confundir fluidez estadística con acceso al significado.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Gebru et al. (2018), Datasheets for Datasets
- Bender y Koller (2020), significado y forma


## 4. Intuición

Un corpus «de toda la web» no es un espejo del mundo: es un espejo de quién publica en la web. Y cuando se limpia con una lista de palabras prohibidas, el filtro no cae por igual sobre todos —cae más fuerte sobre las comunidades que reapropian los términos que la lista bloquea.


## 5. Concepto mínimo

```text
Antes del filtro:  mayoritaria 94,0 %  ·  minoritaria_A 4,0 %  ·  minoritaria_B 2,0 %
Filtro por lista:  retira 1,3 %          ·  retira 47,5 %          ·  retira 15,0 %
Después:           96,07 %               ·  2,17 %                ·  1,76 %

El filtro se aplica igual a todos y NO afecta igual a todos.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('stochastic_parrots', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué porcentaje pierde cada comunidad con el mismo filtro?
2. ¿Sube o baja la cuota de la mayoritaria tras «limpiar»?
3. ¿Qué verá quien audite el corpus con una muestra de 20 documentos?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('stochastic_parrots', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('stochastic_parrots', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

La comunidad minoritaria A pierde el 47,5 % y la mayoritaria el 1,3 %. Tras filtrar, la cuota de la mayoritaria **sube** 2,07 puntos: una operación presentada como higiene técnica redistribuye quién está representado. Y una muestra de 20 documentos contiene 20 de la mayoritaria: quien audite así no verá siquiera que existen las otras.


## 10. Comentario pedagógico

El artículo se cita mucho y se lee poco. Su argumento no es «los modelos grandes son malos», sino que tres costes —ambiental, de representación y de atribución de comprensión— no aparecen en ninguna tabla de resultados. Documentar el corpus antes de entrenar es la propuesta concreta, y es la que menos se ha adoptado.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer «loro estocástico» como un insulto o como una tesis sobre capacidades.


In [ ]:
print('La tesis es linguistica: el modelo ordena FORMAS por probabilidad.')
print('No es una prediccion sobre que tareas podra resolver ni sobre su techo.')
print('Confundir las dos cosas convierte un argumento discutible en un eslogan.')

## 12. Corrección

Lo que el artículo sí sostiene, separado de lo que no:


In [ ]:
afirmaciones = {'sostiene': 'un corpus grande por conveniencia no es representativo',
                'sostiene_2': 'el filtrado por lista silencia a quien reapropia terminos',
                'sostiene_3': 'el coste de entrenar no aparece en la tabla de resultados',
                'no_sostiene': 'que exista un techo de capacidades demostrado'}
show(afirmaciones)

## 13. Desafío guiado

Calcula con los datos del motor cuántos puntos de cuota gana la comunidad mayoritaria y explica por qué una operación «neutral» puede tener un efecto que no lo es.


In [ ]:
r = run_paper_lab('stochastic_parrots', seed=3)['result']
show(r)

## 14. Desafío autónomo

Toma un conjunto de datos que uses y escríbele una hoja de datos: de dónde viene, qué poblaciones representa, qué filtros se le aplicaron y a quién dejaron fuera. Después estima qué decisiones de tu sistema cambiarían si esa composición fuese otra.


## 15. Evidencia de aprendizaje

Guarda la tabla de cuotas antes y después del filtro y tu separación entre lo que el artículo sostiene y lo que se le atribuye.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P61_stochastic_parrots/README.md) · evaluación formal: [`assessments/papers/P61_stochastic_parrots.md`](../../assessments/papers/P61_stochastic_parrots.md)


## 16. Cierre

Si el corpus condiciona lo que un modelo puede decir, la siguiente pregunta es qué condiciona lo que un benchmark puede medir.


## 17. Conexión con el siguiente hito

- P50
- P52

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
